In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Loading Data

In [2]:
# Load the dataset
df = pd.read_csv('../../../EXAM/DB_data_2025-10.csv', sep=',')
df.sample(5)

,station_name,xml_station_name,eva,train_name,final_destination_station,delay_in_min,time,is_canceled,train_type,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,id
1134275,Essen Hbf,Essen Hbf,8000098,S 1,Düsseldorf Hbf,0,2025-10-18 18:06:00,False,S,NaN,3,2025-10-18 18:05:00,2025-10-18 18:05:00,2025-10-18 18:06:00,2025-10-18 18:06:00,-5128726315011753809-2510181759-3
893854,Berlin Ostbahnhof,Berlin Ostbahnhof,8010255,RE 8,Wustermark,11,2025-10-15 02:13:00,False,RE,NaN,3,2025-10-15 02:01:00,2025-10-15 02:07:00,2025-10-15 02:02:00,2025-10-15 02:13:00,-6518168345394142103-2510150144-3
855952,Hamburg Hbf,Hamburg Hbf,8002549,RB 81,Bargteheide,1,2025-10-14 13:12:00,False,RB,7.972017e+18,1,NaN,NaN,2025-10-14 13:11:00,2025-10-14 13:12:00,7972016694215136640-2510141311-1
336248,Berlin Potsdamer Platz,Berlin Potsdamer Platz (S),8089032,S 2,Berlin-Buch,0,2025-10-06 12:24:00,False,S,8.398512e+18,10,2025-10-06 12:23:00,2025-10-06 12:24:00,2025-10-06 12:24:00,2025-10-06 12:24:00,8398512135701862014-2510061202-10
805055,Halle (Saale) Hbf,Halle(Saale)Hbf,8010159,S 8,Halle (Saale) Hbf,3,2025-10-13 17:43:00,False,S,5.144924e+18,6,2025-10-13 17:40:00,2025-10-13 17:43:00,NaN,NaN,5144924022936222553-2510131720-6


## Data Inspection and Cleaning

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1989180 entries, 0 to 1989179
Data columns (total 16 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   station_name               object 
 1   xml_station_name           object 
 2   eva                        int64  
 3   train_name                 object 
 4   final_destination_station  object 
 5   delay_in_min               int64  
 6   time                       object 
 7   is_canceled                bool   
 8   train_type                 object 
 9   train_line_ride_id         float64
 10  train_line_station_num     int64  
 11  arrival_planned_time       object 
 12  arrival_change_time        object 
 13  departure_planned_time     object 
 14  departure_change_time      object 
 15  id                         object 
dtypes: bool(1), float64(1), int64(3), object(11)
memory usage: 229.5+ MB


In [4]:
# Create table

def format_example_values(series):
    vals = series.dropna().unique()[:5]
    
    formatted = []
    for v in vals:
        if isinstance(v, (int, float, np.number)):
            formatted.append(f"{v:.2f}")
        else:
            formatted.append(str(v))
    return ", ".join(formatted)

summary_df = pd.DataFrame({
    "Feature/variable": df.columns,
    "Data type": df.dtypes.values.astype(str),
    "Number of Unique values": [df[col].nunique() for col in df.columns],
    "Missing values": [df[col].isna().sum() for col in df.columns],
    "Example values": [format_example_values(df[col]) for col in df.columns]
    
})

# Show all rows without truncation
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    print(summary_df.to_markdown(index=False))

| Feature/variable          | Data type   |   Number of Unique values |   Missing values | Example values                                                                                                                                                            |
|:--------------------------|:------------|--------------------------:|-----------------:|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| station_name              | object      |                       107 |                0 | Hamburg Hbf, München Ost, Köln Hbf, Stuttgart Hbf, Duisburg Hbf                                                                                                           |
| xml_station_name          | object      |                       130 |                0 | Hamburg Hbf, München Ost, Köln Hbf, Stuttgart Hbf, Duisburg Hbf                                                         

In [5]:
# Create table for numeric columns
numeric_cols = ["delay_in_min"]

# Basic descriptive statistics
desc = df[numeric_cols].describe().rename(index={
    "50%": "50%",
    "25%": "25%",
    "75%": "75%"
})

# Add variance
desc.loc["Variance"] = df[numeric_cols].var()

# Add dispersion index = variance / mean
dispersion_index = df[numeric_cols].var() / df[numeric_cols].mean()
desc.loc["Dispersion index (Variance / Mean)"] = dispersion_index

# Reorder rows 
row_order = [
    "count", "mean", "std", "min", "25%", "50%", "75%", "max",
    "Variance", "Dispersion index (Variance / Mean)"
]
desc = desc.loc[row_order]

# Convert to Markdown
md_table = desc.to_markdown(tablefmt="pipe", floatfmt=".2f")
print(md_table)

|                                    |   delay_in_min |
|:-----------------------------------|---------------:|
| count                              |     1989180.00 |
| mean                               |           4.89 |
| std                                |          10.95 |
| min                                |       -1435.00 |
| 25%                                |           0.00 |
| 50%                                |           1.00 |
| 75%                                |           5.00 |
| max                                |         386.00 |
| Variance                           |         119.79 |
| Dispersion index (Variance / Mean) |          24.50 |


In [6]:
# Descriptive table for categorical variables

categorical_cols = df.select_dtypes(include=['object']).columns
cat_desc = pd.DataFrame(index=categorical_cols, columns=[
    "Unique Values"
])

for col in categorical_cols:
    counts = df[col].value_counts(dropna=False)
    most_freq_val = counts.idxmax()
    freq = counts.max()
    perc = freq / counts.sum() * 100

    cat_desc.loc[col] = [
        df[col].nunique()
    ]

# Convert to Markdown
md_table_cat = cat_desc.to_markdown(tablefmt="pipe", floatfmt=".2f")
print(md_table_cat)

|                           |   Unique Values |
|:--------------------------|----------------:|
| station_name              |             107 |
| xml_station_name          |             130 |
| train_name                |            1573 |
| final_destination_station |            1353 |
| time                      |           44530 |
| train_type                |              55 |
| arrival_planned_time      |           44338 |
| arrival_change_time       |           44365 |
| departure_planned_time    |           44353 |
| departure_change_time     |           44354 |
| id                        |         1989180 |


In [7]:
# check mising values
missing_counts = df.isna().sum()
print(missing_counts)

station_name                      0
xml_station_name                  0
eva                               0
train_name                        0
final_destination_station         0
delay_in_min                      0
time                              0
is_canceled                       0
train_type                        0
train_line_ride_id           988616
train_line_station_num            0
arrival_planned_time         446738
arrival_change_time          446699
departure_planned_time       445857
departure_change_time        445805
id                                0
dtype: int64


In [8]:
# check for duplicates 
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

Number of duplicate rows: 0


In [9]:
# delete duplicates
df.drop_duplicates(inplace=True)
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

Number of duplicate rows: 0


In [10]:
# compare all values in station_name and xml_station_name and output the differences

station_names = set(df["station_name"].unique())
xml_station_names = set(df["xml_station_name"].unique())
differences = station_names.symmetric_difference(xml_station_names)
print("Values in station_name but not in xml_station_name:", station_names - xml_station_names)
print("Values in xml_station_name but not in station_name:", xml_station_names - station_names)

Values in station_name but not in xml_station_name: {'Landshut (Bay) Hbf', 'Ludwigshafen (Rhein) Hbf', 'Oldenburg (Oldb) Hbf', 'Berlin Hauptbahnhof', 'Halle (Saale) Hbf', 'Neustadt (Weinstr) Hbf', 'Münster (Westf) Hbf', 'Singen (Hohentwiel)', 'Hamm (Westf) Hbf', 'Frankfurt (Main) Süd', 'Frankfurt (Oder)', 'Frankfurt (Main) Hbf', 'Freiburg (Breisgau) Hbf', 'Fürth (Bay) Hbf'}
Values in xml_station_name but not in station_name: {'München Hbf Gl.5-10', 'München Hbf (tief)', 'Hamburg-Harburg(S)', 'Leipzig Hbf (tief)', 'Frankfurt(Main)Hbf', 'Stuttgart Hbf (tief)', 'Hamburg Hbf (S-Bahn)', 'Singen(Hohentwiel)', 'Hamburg-Altona(S)', 'Halle(Saale)Hbf', 'Berlin Zoologischer Garten (S)', 'Frankfurt(Oder)', 'Münster(Westf)Hbf', 'München Hbf Gl.27-36', 'Berlin Hbf', 'Berlin Ostkreuz (S)', 'Neustadt(Weinstr)Hbf', 'Hamm(Westf)Hbf', 'Berlin-Spandau (S)', 'Fürth(Bay)Hbf', 'Flughafen BER (S-Bahn)', 'Berlin Potsdamer Platz (S)', 'Potsdam Hbf (S)', 'Landshut(Bay)Hbf', 'Berlin Hbf (S-Bahn)', 'Berlin Südkreu

Since station_name column has more clean data, while xml_station_name has some irrelevant data ( duplicates, platform numbers etc.), it is decided to use station_name in analysis.

In [11]:
# Drop extra columns

df.drop(columns=["id", "xml_station_name", "eva", "train_line_ride_id"], inplace=True)


Since destination_name has many unique values, we check for possible duplicates and misspellings

In [ ]:
# Create a new column for cleaned destination, remove spaces and dashes
df["cleaned_destination"] = (df["final_destination_station"]
.str.replace(" ", "", regex=False)
.str.replace("-", "", regex=False))

# Remove the S-Bahn marker
df["cleaned_destination"] = (df["cleaned_destination"]
.str.replace("(S)", "", regex=False)
.str.replace("(s)", "", regex=False))

# Fix specific duplicates, swap "(Rh)" for "(Rhein)"
df["cleaned_destination"] = (df["cleaned_destination"]
.str.replace("Terminal", "", regex=False)
.str.replace("(Rh)", "(Rhein)", regex=False))

# Remove "Gl." and track numbers
df["cleaned_destination"] = df["cleaned_destination"].str.replace(r"Gl\.\d*", "", regex=True)

# Print the new number of values
new_unique_count = df["cleaned_destination"].nunique()
print("New number of unique stations:", new_unique_count)

# Insert cleaned column
cleaned_column_data = df.pop("cleaned_destination")
insert_position = df.columns.get_loc("final_destination_station")
df.insert(insert_position, "cleaned_destination", cleaned_column_data)

New number of unique stations: 1320


In [13]:
# Check that there are no rows with both times of arrival and departure missing

missing_both_count = (df["arrival_planned_time"].isna() & df["departure_planned_time"].isna()).sum()
print("Number of rows missing both arrival and departure:", missing_both_count)

Number of rows missing both arrival and departure: 0


In [14]:
df

,station_name,train_name,cleaned_destination,final_destination_station,delay_in_min,time,is_canceled,train_type,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time
0,Hamburg Hbf,ME RB41,HamburgHbf,Hamburg Hbf,34,2025-10-01 00:00:00,False,ME,15,2025-09-30 23:26:00,2025-10-01 00:00:00,NaN,NaN
1,München Ost,S 1,Freising,Freising,7,2025-10-01 00:00:00,False,S,2,2025-09-30 23:52:00,2025-09-30 23:56:00,2025-09-30 23:53:00,2025-10-01 00:00:00
2,Köln Hbf,S 11,Norf,Norf,16,2025-10-01 00:00:00,True,S,8,2025-09-30 23:43:00,2025-10-01 00:25:00,2025-09-30 23:44:00,2025-10-01 00:00:00
3,Stuttgart Hbf,RE 8,StuttgartHbf,Stuttgart Hbf,8,2025-10-01 00:00:00,False,RE,5,2025-09-30 23:52:00,2025-10-01 00:00:00,NaN,NaN
4,Duisburg Hbf,RE 3,DüsseldorfHbf,Düsseldorf Hbf,7,2025-10-01 00:00:00,False,RE,9,2025-09-30 23:50:00,2025-10-01 00:00:00,2025-09-30 23:53:00,2025-10-01 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1989175,Berlin-Spandau,RE 4,BerlinHbf,Berlin Hbf,7,2025-10-31 23:59:00,False,RE,9,2025-10-31 23:50:00,2025-10-31 23:57:00,2025-10-31 23:52:00,2025-10-31 23:59:00
1989176,Köln Messe/Deutz,RB 25,Gummersbach,Gummersbach,2,2025-10-31 23:59:00,False,RB,3,2025-10-31 23:56:00,2025-10-31 23:58:00,2025-10-31 23:57:00,2025-10-31 23:59:00
1989177,Bietigheim-Bissingen,MEX 18,BietigheimBissingen,Bietigheim-Bissingen,1,2025-10-31 23:59:00,False,MEX,13,2025-10-31 23:58:00,2025-10-31 23:59:00,NaN,NaN
1989178,Bielefeld Hbf,NWB RB75,BielefeldHbf,Bielefeld Hbf,6,2025-10-31 23:59:00,False,NWB,19,2025-10-31 23:53:00,2025-10-31 23:59:00,NaN,NaN
